# Multi-Head Attention

## Learning Objectives
- Understand the rationale and implementation of multi-head attention.
- Learn how multiple attention heads provide diverse feature extraction.

## Introduction
Multi-head attention splits queries, keys, and values into multiple smaller sets processed in parallel, capturing different aspects of relationships in sequences.

## Core Concepts
- **Multiple heads:** Parallel attention mechanisms.
- **Concatenation:** Combine heads outputs.
- **Linear projections:** Query, key, value projections per head.

## Example
Implement simplified multi-head attention.

In [ ]:
import tensorflow as tf

class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.depth = d_model // num_heads
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q):
        batch_size = tf.shape(q)[0]
        Q = self.wq(q)
        K = self.wk(k)
        V = self.wv(v)
        Q = self.split_heads(Q, batch_size)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)
        
        matmul_qk = tf.matmul(Q, K, transpose_b=True)
        dk = tf.cast(tf.shape(K)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, V)
        output = tf.transpose(output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(output, (batch_size, -1, self.num_heads * self.depth))
        final_output = self.dense(concat_attention)
        return final_output

# Create example
mha = MultiHeadAttention(d_model=64, num_heads=8)
dummy_q = tf.random.uniform((1, 10, 64))
dummy_k = tf.random.uniform((1, 10, 64))
dummy_v = tf.random.uniform((1, 10, 64))
output = mha(dummy_v, dummy_k, dummy_q)
print(f"Multi-Head Attention output shape: {output.shape}")

## Exercise
Increase number of heads and observe how output shape changes.

In [ ]:
# Your code here

## Summary
- Multi-head attention enables models to attend to information from multiple representation subspaces.
- This enhances the expressive power of Transformers.


## Further Reading
- [Multi-Head Attention Explained](https://jalammar.github.io/illustrated-transformer/#multi-head-attention)
- [TensorFlow MultiHeadAttention API](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MultiHeadAttention)
